# Notebook 3 — Atlantic Canada CO₂ Storage COS Mapping

## Objective

Explore the Geological Survey of Canada Open File 8996 geospatial datasets for
Atlantic Canadian geological CO₂ storage potential.

This notebook will:

1. Inventory the Mesozoic–Cenozoic and Upper Paleozoic COS shapefiles.
2. Inspect CRS, geometry, fields, feature counts, and spatial extent.
3. Determine the meaning and structure of the Chance of Success (COS) attributes.
4. Compare schemas across formations and geological intervals.
5. Identify which source fields should be preserved in a harmonized CanCO₂Re/CANCARB
   geological-storage database.
6. Reproject harmonized Silver geometries to the CanCO₂Re project standard CRS, EPSG:3978 (NAD83 / Canada Atlas Lambert).
7. Identify semantic limitations before combining these data with storage-capacity,
   injectivity, project, basin, or regulatory datasets.

## Source

Carey, J.S., Skinner, C.H., Giles, P.S., Durling, P., Plourde, A.P.,
Jauer, C., and Desroches, K. (2023).
*Preliminary assessment of geological carbon-storage potential of Atlantic Canada*.
Geological Survey of Canada, Open File 8996.

Publication:
https://ostrnrcan-dostrncan.canada.ca/entities/publication/b6140ab8-c5fd-4ad2-b9ed-65c32ff2bb17

ZIP archive:
https://ostrnrcan-dostrncan.canada.ca/bitstreams/5a707088-2799-4c70-94c3-656f80256612/download

DOI:
https://doi.org/10.4095/332145

## CRS policy

The original CRS of each Bronze shapefile is preserved in `source_crs` for provenance.
The harmonized CanCO₂Re Silver layer is standardized to EPSG:3978 (NAD83 / Canada Atlas Lambert), consistent with the CanCO₂Re GIS Data Management Protocol.


In [1]:
# ---------------------------------------------------------------------------
# Imports and paths
# ---------------------------------------------------------------------------

from pathlib import Path

import geopandas as gpd
import pandas as pd
import sqlite3
from pyproj import CRS

# ---------------------------------------------------------------------------
# Source directories
# ---------------------------------------------------------------------------

ROOT = Path(
    r"C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage"
)

MESO_DIR = ROOT / "Mesozoic-Cenozoic COS Mapping"
PALEO_DIR = ROOT / "Upper Paleozoic COS Mapping"

REPORT_PATH = ROOT / "of_8996.pdf"
README_PATH = ROOT / "of_8996_readme.rtf"

# ---------------------------------------------------------------------------
# Source metadata
# ---------------------------------------------------------------------------

PUBLICATION_URL = (
    "https://ostrnrcan-dostrncan.canada.ca/entities/publication/"
    "b6140ab8-c5fd-4ad2-b9ed-65c32ff2bb17"
)

ZIP_URL = (
    "https://ostrnrcan-dostrncan.canada.ca/bitstreams/"
    "5a707088-2799-4c70-94c3-656f80256612/download"
)

DOI = "https://doi.org/10.4095/332145"

CANCO2RE_CRS = "EPSG:3978"

print(f"Root:                 {ROOT}")
print(f"Mesozoic-Cenozoic:    {MESO_DIR.exists()}")
print(f"Upper Paleozoic:      {PALEO_DIR.exists()}")
print(f"Report:                {REPORT_PATH.exists()}")
print(f"README:                {README_PATH.exists()}")
print(f"CanCO2Re Silver CRS:   {CANCO2RE_CRS}")

Root:                 C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage
Mesozoic-Cenozoic:    True
Upper Paleozoic:      True
Report:                True
README:                True
CanCO2Re Silver CRS:   EPSG:3978


In [2]:
# ---------------------------------------------------------------------------
# Inventory source shapefiles
# ---------------------------------------------------------------------------

source_groups = {
    "Mesozoic-Cenozoic": MESO_DIR,
    "Upper Paleozoic": PALEO_DIR,
}

inventory_records = []

for geological_group, directory in source_groups.items():

    shapefiles = sorted(directory.glob("*.shp"))

    for shp_path in shapefiles:

        gdf = gpd.read_file(shp_path)

        inventory_records.append(
            {
                "geological_group": geological_group,
                "dataset": shp_path.stem,
                "filename": shp_path.name,
                "features": len(gdf),
                "columns": len(gdf.columns),
                "crs": str(gdf.crs),
                "geometry_types": ", ".join(
                    sorted(gdf.geometry.geom_type.dropna().unique())
                ),
                "missing_geometry": int(gdf.geometry.isna().sum()),
                "path": str(shp_path),
            }
        )

inventory = (
    pd.DataFrame(inventory_records)
    .sort_values(["geological_group", "dataset"])
    .reset_index(drop=True)
)

display(inventory)

print("\nDataset counts:")
display(
    inventory.groupby("geological_group")
    .size()
    .rename("datasets")
    .to_frame()
)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,geological_group,dataset,filename,features,columns,crs,geometry_types,missing_geometry,path
0,Mesozoic-Cenozoic,ALBIAN_LOGAN_CANYON_TOTAL,ALBIAN_LOGAN_CANYON_TOTAL.shp,17,10,EPSG:26720,"MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
1,Mesozoic-Cenozoic,BARREMIAN_UPPER_MISSISAUGA_TOTAL,BARREMIAN_UPPER_MISSISAUGA_TOTAL.shp,10,10,EPSG:26720,Polygon,0,C:\Users\aviga\Research\potential data\Storage...
2,Mesozoic-Cenozoic,BERRIASIAN_LOWER_MISSISAUGA_TOTAL,BERRIASIAN_LOWER_MISSISAUGA_TOTAL.shp,12,10,EPSG:26720,"MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
3,Mesozoic-Cenozoic,Bjarni,Bjarni.shp,407,11,"PROJCS[""NAD83(CSRS)v2 / Quebec Lambert"",GEOGCS...","MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
4,Mesozoic-Cenozoic,Fundy,Fundy.shp,43,11,EPSG:8082,"MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
5,Mesozoic-Cenozoic,Gudrid,Gudrid.shp,812,12,"PROJCS[""NAD83(CSRS)v2 / Quebec Lambert"",GEOGCS...","MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
6,Mesozoic-Cenozoic,LATE_ALBIAN_CREE_TOTAL,LATE_ALBIAN_CREE_TOTAL.shp,8,11,EPSG:26720,Polygon,0,C:\Users\aviga\Research\potential data\Storage...
7,Mesozoic-Cenozoic,Leif,Leif.shp,472,15,"PROJCS[""NAD83(CSRS)v2 / Quebec Lambert"",GEOGCS...",Polygon,0,C:\Users\aviga\Research\potential data\Storage...
8,Mesozoic-Cenozoic,UPPER_JURASSIC_MOHAWK_MIC_MAC_TOTAL,UPPER_JURASSIC_MOHAWK_MIC_MAC_TOTAL.shp,17,10,EPSG:26720,"MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...
9,Mesozoic-Cenozoic,VALANGINIAN_HAUTERIVIAN_MID_MISSISAUGA_TOTAL,VALANGINIAN_HAUTERIVIAN_MID_MISSISAUGA_TOTAL.shp,13,12,EPSG:26720,"MultiPolygon, Polygon",0,C:\Users\aviga\Research\potential data\Storage...



Dataset counts:


,datasets
geological_group,
Mesozoic-Cenozoic,10
Upper Paleozoic,5


In [3]:
# ---------------------------------------------------------------------------
# Detailed field inventory by dataset
# ---------------------------------------------------------------------------

for geological_group, directory in source_groups.items():

    print("\n" + "=" * 80)
    print(geological_group)
    print("=" * 80)

    for shp_path in sorted(directory.glob("*.shp")):

        gdf = gpd.read_file(shp_path)

        print(f"\n{shp_path.stem}")
        print("-" * len(shp_path.stem))

        print(f"Features: {len(gdf):,}")
        print(f"CRS:      {gdf.crs}")
        print(f"Geometry: {gdf.geometry.geom_type.value_counts(dropna=False).to_dict()}")

        display(
            pd.DataFrame(
                {
                    "dtype": gdf.dtypes.astype(str),
                    "non_null": gdf.notna().sum(),
                    "null": gdf.isna().sum(),
                    "unique": gdf.nunique(dropna=True),
                }
            )
        )


Mesozoic-Cenozoic

ALBIAN_LOGAN_CANYON_TOTAL
-------------------------
Features: 17
CRS:      EPSG:26720
Geometry: {'Polygon': 15, 'MultiPolygon': 2}


,dtype,non_null,null,unique
FID_8_2_11,int64,17,0,5
Id,int64,17,0,1
COS_RES,float64,17,0,4
FID_8_2_12,int64,17,0,5
Id_1,int64,17,0,1
COS_SEAL,float64,17,0,5
Shape_Leng,float64,17,0,14
Shape_Area,float64,17,0,14
Total_COS,float64,17,0,10
geometry,geometry,17,0,14



BARREMIAN_UPPER_MISSISAUGA_TOTAL
--------------------------------
Features: 10
CRS:      EPSG:26720
Geometry: {'Polygon': 10}


,dtype,non_null,null,unique
FID_8_2_9_,int64,10,0,3
Id,int64,10,0,1
COS_RES,float64,10,0,3
FID_8_2_91,int64,10,0,6
Id_1,int64,10,0,1
COS_SEAL,float64,10,0,4
Shape_Leng,float64,10,0,10
Shape_Area,float64,10,0,10
Total_COS,float64,10,0,8
geometry,geometry,10,0,10



BERRIASIAN_LOWER_MISSISAUGA_TOTAL
---------------------------------
Features: 12
CRS:      EPSG:26720
Geometry: {'Polygon': 11, 'MultiPolygon': 1}


...

,dtype,non_null,null,unique
COS_Seal,float64,712,0,4
Play_Descr,str,712,0,8
Descriptio,str,712,0,1
COS_Trap,float64,712,0,8
Play_des_1,str,712,0,21
COS_Reserv,float64,712,0,14
Play312_De,str,712,0,39
Shape_Leng,float64,712,0,702
Shape_Area,float64,712,0,702
CCOS_CCUS,float64,712,0,71



Magdalen_Horton
---------------
Features: 691
CRS:      PROJCS["NAD83(CSRS)v2 / Quebec Lambert",GEOGCS["NAD83(CSRS)",DATUM["NAD83_Canadian_Spatial_Reference_System",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6140"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",44],PARAMETER["central_meridian",-68.5],PARAMETER["standard_parallel_1",46],PARAMETER["standard_parallel_2",60],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry: {'Polygon': 480, 'MultiPolygon': 211}


,dtype,non_null,null,unique
OBJECTID,int64,691,0,691
COS_Seal,float64,691,0,4
Play352_De,str,691,0,5
COS_Trap,float64,691,0,3
Play352_Tr,str,691,0,11
COS_Reserv,float64,691,0,10
Play352__2,str,691,0,14
Shape_Leng,float64,691,0,691
Shape_Area,float64,691,0,691
COS_CCUS,float64,691,0,47



Pictou
------
Features: 1,166
CRS:      PROJCS["NAD83(CSRS)v2 / Quebec Lambert",GEOGCS["NAD83(CSRS)",DATUM["NAD83_Canadian_Spatial_Reference_System",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6140"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",44],PARAMETER["central_meridian",-68.5],PARAMETER["standard_parallel_1",46],PARAMETER["standard_parallel_2",60],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry: {'Polygon': 710, 'MultiPolygon': 456}


,dtype,non_null,null,unique
COS_Seal,float64,1166,0,5
Play_Descr,str,1166,0,5
COS_Trap,float64,1166,0,6
Play_des_1,str,1166,0,11
Descriptio,str,1166,0,1
COS_Reserv,float64,1166,0,14
Play_Des_2,str,1166,0,26
Shape_Leng,float64,1166,0,1070
Shape_Area,float64,1166,0,1070
CCOS_CCUS,float64,1166,0,63


In [4]:
# ---------------------------------------------------------------------------
# Compare attribute schemas across all COS datasets
# ---------------------------------------------------------------------------

schema_records = []

for geological_group, directory in source_groups.items():

    for shp_path in sorted(directory.glob("*.shp")):

        gdf = gpd.read_file(shp_path)

        for column in gdf.columns:

            if column == gdf.geometry.name:
                continue

            schema_records.append(
                {
                    "geological_group": geological_group,
                    "dataset": shp_path.stem,
                    "column": column,
                    "dtype": str(gdf[column].dtype),
                    "non_null": int(gdf[column].notna().sum()),
                    "null": int(gdf[column].isna().sum()),
                    "unique": int(gdf[column].nunique(dropna=True)),
                }
            )

schema_inventory = pd.DataFrame(schema_records)

# How widely is each field used?
field_summary = (
    schema_inventory
    .groupby("column")
    .agg(
        dataset_count=("dataset", "nunique"),
        dtypes=("dtype", lambda x: ", ".join(sorted(set(x)))),
        datasets=("dataset", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
    .sort_values(
        ["dataset_count", "column"],
        ascending=[False, True],
    )
)

display(field_summary)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,column,dataset_count,dtypes,datasets
60,Shape_Area,15,float64,"ALBIAN_LOGAN_CANYON_TOTAL, BARREMIAN_UPPER_MIS..."
62,Shape_Leng,15,float64,"ALBIAN_LOGAN_CANYON_TOTAL, BARREMIAN_UPPER_MIS..."
3,COS_RES,6,float64,"ALBIAN_LOGAN_CANYON_TOTAL, BARREMIAN_UPPER_MIS..."
4,COS_Reserv,6,float64,"Bjarni, Gudrid, Leif, Magdalen_Cumberland, Mag..."
5,COS_SEAL,6,float64,"ALBIAN_LOGAN_CANYON_TOTAL, BARREMIAN_UPPER_MIS..."
...,...,...,...,...
63,TCOS_CCUS,1,float64,Bjarni
66,Tr_COS,1,float64,Cumberland_non_Magdalen
67,Tr_COS_Des,1,str,Cumberland_non_Magdalen
68,Trap_COS,1,float64,Fundy


In [5]:
# ---------------------------------------------------------------------------
# Inspect low-cardinality source attributes
# ---------------------------------------------------------------------------

for geological_group, directory in source_groups.items():

    print("\n" + "=" * 90)
    print(geological_group)
    print("=" * 90)

    for shp_path in sorted(directory.glob("*.shp")):

        gdf = gpd.read_file(shp_path)

        print(f"\n{shp_path.stem}")
        print("-" * len(shp_path.stem))

        attribute_columns = [
            column
            for column in gdf.columns
            if column != gdf.geometry.name
        ]

        for column in attribute_columns:

            unique_count = gdf[column].nunique(dropna=True)

            print(
                f"{column:<20} "
                f"dtype={str(gdf[column].dtype):<10} "
                f"unique={unique_count:<5} "
                f"null={gdf[column].isna().sum()}"
            )

            if unique_count <= 20:
                print(
                    "   values:",
                    gdf[column]
                    .dropna()
                    .value_counts()
                    .to_dict()
                )


Mesozoic-Cenozoic

ALBIAN_LOGAN_CANYON_TOTAL
-------------------------
FID_8_2_11           dtype=int64      unique=5     null=0
   values: {4: 7, 2: 4, 0: 3, 1: 2, 3: 1}
Id                   dtype=int64      unique=1     null=0
   values: {0: 17}
COS_RES              dtype=float64    unique=4     null=0
   values: {0.2: 7, 0.45: 6, 0.75: 3, 0.15: 1}
FID_8_2_12           dtype=int64      unique=5     null=0
   values: {1: 6, 3: 5, 0: 4, 2: 1, 4: 1}
Id_1                 dtype=int64      unique=1     null=0
   values: {0: 17}
COS_SEAL             dtype=float64    unique=5     null=0
   values: {0.15: 6, 0.75: 5, 0.65: 4, 0.25: 1, 0.5: 1}
Shape_Leng           dtype=float64    unique=14    null=0
   values: {22459.0744009: 2, 35150.0491473: 2, 126131.50255: 2, 758998.180363: 1, 1538202.15794: 1, 1015070.13977: 1, 141034.366237: 1, 2609816.85105: 1, 146927.698527: 1, 28724.6580526: 1, 1965315.49573: 1, 283508.132089: 1, 849562.49263: 1, 1166092.93191: 1}
Shape_Area           dtype=float64 

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(



Bjarni
------
COS_Seal             dtype=float64    unique=12    null=0
   values: {0.55: 141, 0.45: 71, 0.85: 43, 0.05: 41, 0.5: 29, 0.8: 22, 0.3: 17, 0.9: 16, 0.75: 8, 0.2: 7, 0.7: 7, 0.01: 5}
Play062_Se           dtype=str        unique=32    null=0
Play100_Tr           dtype=str        unique=126   null=0
Type                 dtype=str        unique=4     null=0
   values: {'struc': 334, 'struc/strat': 57, 'strat': 14, 'strat?': 2}
COS_Trap             dtype=float64    unique=16    null=0
   values: {0.8: 178, 0.5: 43, 0.65: 40, 0.1: 30, 0.85: 25, 0.9: 21, 0.75: 12, 0.55: 11, 0.6: 11, 0.45: 9, 0.2: 8, 0.01: 7, 0.25: 6, 0.3: 4, 0.05: 1, 0.7: 1}
Play100_Re           dtype=str        unique=37    null=234
COS_Reserv           dtype=float64    unique=13    null=0
   values: {0.5: 236, 0.75: 41, 0.55: 29, 0.65: 24, 0.8: 22, 0.05: 21, 0.01: 9, 0.6: 9, 0.85: 5, 0.1: 4, 1.0: 4, 0.7: 2, 0.45: 1}
Shape_Leng           dtype=float64    unique=298   null=0
Shape_Area           dtype=float64   

In [6]:
# ---------------------------------------------------------------------------
# Inspect representative source records
# ---------------------------------------------------------------------------

representative_files = [
    MESO_DIR / "ALBIAN_LOGAN_CANYON_TOTAL.shp",
    MESO_DIR / "Bjarni.shp",
    MESO_DIR / "Fundy.shp",
    PALEO_DIR / "Magdalen_Horton.shp",
]

for shp_path in representative_files:

    gdf = gpd.read_file(shp_path)

    print("\n" + "=" * 90)
    print(shp_path.stem)
    print("=" * 90)

    print("CRS:")
    print(gdf.crs)

    print("\nColumns:")
    print(gdf.columns.tolist())

    print("\nFirst 10 attribute records:")
    display(
        gdf.drop(columns="geometry").head(10)
    )


ALBIAN_LOGAN_CANYON_TOTAL
CRS:
EPSG:26720

Columns:
['FID_8_2_11', 'Id', 'COS_RES', 'FID_8_2_12', 'Id_1', 'COS_SEAL', 'Shape_Leng', 'Shape_Area', 'Total_COS', 'geometry']

First 10 attribute records:


,FID_8_2_11,Id,COS_RES,FID_8_2_12,Id_1,COS_SEAL,Shape_Leng,Shape_Area,Total_COS
0,0,0,0.75,2,0,0.25,7.589982e+05,1.008145e+10,0.1875
1,0,0,0.75,3,0,0.75,1.538202e+06,2.621282e+10,0.5625
2,0,0,0.75,4,0,0.50,1.015070e+06,9.016988e+09,0.3750
3,1,0,0.45,1,0,0.15,1.410344e+05,1.198301e+08,0.0675
4,1,0,0.45,3,0,0.75,2.609817e+06,6.371534e+10,0.3375
5,2,0,0.45,0,0,0.65,1.469277e+05,6.260938e+08,0.2925
6,2,0,0.45,1,0,0.15,2.872466e+04,2.665685e+07,0.0675
7,3,0,0.15,3,0,0.75,1.965315e+06,2.815972e+10,0.1125
8,4,0,0.20,0,0,0.65,2.835081e+05,3.228015e+09,0.1300
9,4,0,0.20,1,0,0.15,8.495625e+05,6.124279e+09,0.0300



Bjarni
CRS:
PROJCS["NAD83(CSRS)v2 / Quebec Lambert",GEOGCS["NAD83(CSRS)",DATUM["NAD83_Canadian_Spatial_Reference_System",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6140"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",44],PARAMETER["central_meridian",-68.5],PARAMETER["standard_parallel_1",46],PARAMETER["standard_parallel_2",60],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Columns:
['COS_Seal', 'Play062_Se', 'Play100_Tr', 'Type', 'COS_Trap', 'Play100_Re', 'COS_Reserv', 'Shape_Leng', 'Shape_Area', 'TCOS_CCUS', 'geometry']

First 10 attribute records:


c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,COS_Seal,Play062_Se,Play100_Tr,Type,COS_Trap,Play100_Re,COS_Reserv,Shape_Leng,Shape_Area,TCOS_CCUS
0,0.20,Thin to absent,Likely far from subcrop.,struc,0.65,NaN,0.5,462631.167148,1.922583e+09,0.06500
1,0.20,Thin to absent,Highly faulted extensional zone on slope. Stru...,struc,0.45,NaN,0.5,509419.253553,6.659481e+09,0.04500
2,0.20,Thin to absent,Limited data coverage. Traps possible.. Likely...,struc/strat,0.65,NaN,0.5,191249.884833,7.390370e+08,0.06500
3,0.55,Variable shale thickness. Shale thins towards ...,Far from subcrop.,struc,0.80,NaN,0.5,149600.619013,1.993663e+08,0.22000
4,0.55,Variable shale thickness. Shale thins towards ...,Highly faulted extensional zone on slope. Stru...,struc,0.45,NaN,0.5,232075.317993,1.222977e+09,0.12375
5,0.45,"If present, it will be more continuous than ot...",Far from subcrop.,struc,0.80,NaN,0.5,619235.355011,5.739839e+09,0.18000
6,0.45,"If present, it will be more continuous than ot...",Poor imaging. Structural trapping possible. Fa...,struc,0.80,NaN,0.5,172554.805530,1.587132e+09,0.18000
7,0.45,"If present, it will be more continuous than ot...",Strong amplitudes likely indicate an igneous s...,struc/strat,0.90,NaN,0.5,449655.238929,9.322764e+09,0.20250
8,0.45,"If present, it will be more continuous than ot...","Poor imaging of deep section, but traps possib...",struc/strat,0.80,NaN,0.5,102893.212484,4.800191e+08,0.18000
9,0.45,"If present, it will be more continuous than ot...",Limited data coverage. Traps possible.. Far fr...,struc/strat,0.80,NaN,0.5,32311.976370,2.942556e+07,0.18000



Fundy
CRS:
EPSG:8082

Columns:
['OBJECTID', 'Trap_COS', 'Trap_COS_D', 'Reservoir_', 'Reservoir1', 'Seal_COS', 'Seal_COS_D', 'Shape_Leng', 'Shape_Area', 'Combined_C', 'geometry']

First 10 attribute records:


,OBJECTID,Trap_COS,Trap_COS_D,Reservoir_,Reservoir1,Seal_COS,Seal_COS_D,Shape_Leng,Shape_Area,Combined_C
0,1,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.8,Good basalt seal,251483.444311,1.960937e+08,0.128
1,2,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.1,Basalt absent,102938.613844,1.364007e+08,0.016
2,3,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.1,Basalt absent,543151.401702,1.215404e+09,0.016
3,4,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.5,Basalt present but base above 850m. CO2 not su...,348759.118557,6.821271e+08,0.080
4,5,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.5,Basalt present but base above 850m. CO2 not su...,70900.525132,1.712538e+08,0.080
5,6,0.2,"Structurally complex northern Fundy margin, lo...",0.80,Top Blomidon 850 - 2550 m,0.5,NaN,100333.775362,1.781664e+08,0.080
6,7,0.2,"Structurally complex northern Fundy margin, lo...",0.05,Depth to basement mapped as < 850 m,0.8,Good basalt seal,236332.893861,6.052278e+08,0.008
7,8,0.2,"Structurally complex northern Fundy margin, lo...",0.05,Depth to basement mapped as < 850 m,0.1,Basalt absent,700470.099174,3.618144e+09,0.001
8,9,0.2,"Structurally complex northern Fundy margin, lo...",0.05,Depth to basement mapped as < 850 m,0.1,Basalt absent,155851.392254,3.174823e+08,0.001
9,10,0.2,"Structurally complex northern Fundy margin, lo...",0.05,Depth to basement mapped as < 850 m,0.5,Basalt present but base above 850m. CO2 not su...,5277.080666,1.273045e+06,0.005



Magdalen_Horton
CRS:
PROJCS["NAD83(CSRS)v2 / Quebec Lambert",GEOGCS["NAD83(CSRS)",DATUM["NAD83_Canadian_Spatial_Reference_System",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6140"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",44],PARAMETER["central_meridian",-68.5],PARAMETER["standard_parallel_1",46],PARAMETER["standard_parallel_2",60],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Columns:
['OBJECTID', 'COS_Seal', 'Play352_De', 'COS_Trap', 'Play352_Tr', 'COS_Reserv', 'Play352__2', 'Shape_Leng', 'Shape_Area', 'COS_CCUS', 'geometry']

First 10 attribute records:


,OBJECTID,COS_Seal,Play352_De,COS_Trap,Play352_Tr,COS_Reserv,Play352__2,Shape_Leng,Shape_Area,COS_CCUS
0,1,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.10,Thickness is greater than 6000 m,786384.043634,7.371322e+09,0.056
1,2,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.15,Thickness is 4000 - 6000 m,160148.296452,3.310681e+08,0.084
2,3,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.15,Thickness is 4000 - 6000 m,219751.680855,3.567174e+08,0.084
3,4,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.15,Thickness is 4000 - 6000 m,105384.405795,1.183536e+08,0.084
4,5,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.60,Thickness is 2000 - 3000 m,44340.064307,5.006906e+07,0.336
5,6,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.35,Thickness is 3000 - 4000 m,71117.208172,6.833784e+07,0.196
6,7,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.35,Thickness is 3000 - 4000 m,241897.249282,3.564573e+08,0.196
7,8,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.15,Thickness is 4000 - 6000 m,25860.498411,9.546802e+06,0.084
8,9,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.15,Thickness is 4000 - 6000 m,14357.338191,2.436495e+06,0.084
9,10,0.7,Thickness greater than 1000 m. Based on summed...,0.8,No nearby Horton outcrop,0.60,Thickness is 2000 - 3000 m,62995.243980,5.429649e+07,0.336


In [7]:
# ---------------------------------------------------------------------------
# Candidate semantic field mappings
# ---------------------------------------------------------------------------

semantic_mappings = {
    "ALBIAN_LOGAN_CANYON_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "Total_COS": "combined_cos",
    },

    "Bjarni": {
        "COS_Reserv": "reservoir_cos",
        "Play100_Re": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play062_Se": "seal_description",
        "COS_Trap": "trap_cos",
        "Play100_Tr": "trap_description",
        "Type": "trap_type",
        "TCOS_CCUS": "combined_cos",
    },

    "Fundy": {
        "Reservoir_": "reservoir_cos",
        "Reservoir1": "reservoir_description",
        "Seal_COS": "seal_cos",
        "Seal_COS_D": "seal_description",
        "Trap_COS": "trap_cos",
        "Trap_COS_D": "trap_description",
        "Combined_C": "combined_cos",
    },

    "Magdalen_Horton": {
        "COS_Reserv": "reservoir_cos",
        "Play352__2": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play352_De": "seal_description",
        "COS_Trap": "trap_cos",
        "Play352_Tr": "trap_description",
        "COS_CCUS": "combined_cos",
    },
}

mapping_records = []

for dataset, mapping in semantic_mappings.items():
    for source_field, semantic_field in mapping.items():
        mapping_records.append(
            {
                "dataset": dataset,
                "source_field": source_field,
                "semantic_field": semantic_field,
            }
        )

mapping_df = pd.DataFrame(mapping_records)

display(mapping_df)

,dataset,source_field,semantic_field
0,ALBIAN_LOGAN_CANYON_TOTAL,COS_RES,reservoir_cos
1,ALBIAN_LOGAN_CANYON_TOTAL,COS_SEAL,seal_cos
2,ALBIAN_LOGAN_CANYON_TOTAL,Total_COS,combined_cos
3,Bjarni,COS_Reserv,reservoir_cos
4,Bjarni,Play100_Re,reservoir_description
5,Bjarni,COS_Seal,seal_cos
6,Bjarni,Play062_Se,seal_description
7,Bjarni,COS_Trap,trap_cos
8,Bjarni,Play100_Tr,trap_description
9,Bjarni,Type,trap_type


In [8]:
# ---------------------------------------------------------------------------
# Mapping-readiness audit
# ---------------------------------------------------------------------------

expected_concepts = {
    "reservoir_cos",
    "reservoir_description",
    "seal_cos",
    "seal_description",
    "trap_cos",
    "trap_description",
    "trap_type",
    "combined_cos",
    "assessment_area",
    "play_description",
}

dataset_fields = {}

for geological_group, directory in source_groups.items():

    for shp_path in sorted(directory.glob("*.shp")):

        gdf = gpd.read_file(shp_path)

        dataset_fields[shp_path.stem] = {
            column
            for column in gdf.columns
            if column != gdf.geometry.name
        }

audit_records = []

for dataset, fields in dataset_fields.items():

    audit_records.append(
        {
            "dataset": dataset,
            "field_count": len(fields),
            "fields": ", ".join(sorted(fields)),
            "has_reservoir_candidate": any(
                x in fields
                for x in [
                    "COS_RES",
                    "COS_Reserv",
                    "Reservoir_",
                    "Res_COS",
                    "Horton_R_8",
                ]
            ),
            "has_seal_candidate": any(
                x in fields
                for x in [
                    "COS_SEAL",
                    "COS_Seal",
                    "Seal_COS",
                    "Sea_COS",
                    "Horton_Sea",
                ]
            ),
            "has_trap_candidate": any(
                x in fields
                for x in [
                    "COS_Trap",
                    "Trap_COS",
                    "Tr_COS",
                    "Horton_Tra",
                ]
            ),
            "has_combined_candidate": any(
                x in fields
                for x in [
                    "Total_COS",
                    "TotalCOS",
                    "TCOS_CCUS",
                    "CCOS",
                    "Combined_C",
                    "CCOS_CCUS",
                    "Horton_TCO",
                    "COS_CCUS",
                ]
            ),
        }
    )

mapping_readiness = (
    pd.DataFrame(audit_records)
    .sort_values("dataset")
    .reset_index(drop=True)
)

display(mapping_readiness)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,dataset,field_count,fields,has_reservoir_candidate,has_seal_candidate,has_trap_candidate,has_combined_candidate
0,ALBIAN_LOGAN_CANYON_TOTAL,9,"COS_RES, COS_SEAL, FID_8_2_11, FID_8_2_12, Id,...",True,True,False,True
1,BARREMIAN_UPPER_MISSISAUGA_TOTAL,9,"COS_RES, COS_SEAL, FID_8_2_91, FID_8_2_9_, Id,...",True,True,False,True
2,BERRIASIAN_LOWER_MISSISAUGA_TOTAL,9,"COS_RES, COS_SEAL, FID_8_2_51, FID_8_2_5_, Id,...",True,True,False,True
3,Bjarni,10,"COS_Reserv, COS_Seal, COS_Trap, Play062_Se, Pl...",True,True,True,True
4,Cumberland_non_Magdalen,10,"CCOS_CCUS, OBJECTID, Res_COS, Res_COS_De, Sea_...",True,True,True,True
5,Fundy,10,"Combined_C, OBJECTID, Reservoir1, Reservoir_, ...",True,True,True,True
6,Gudrid,11,"CCOS, COS_Reserv, COS_Seal, COS_Trap, Name, Pl...",True,True,True,True
7,Horton_non_Magdalen,10,"Horton_R_8, Horton_R_9, Horton_S_1, Horton_Sea...",True,True,True,True
8,LATE_ALBIAN_CREE_TOTAL,10,"COS_RES, COS_SEAL, FID_8_2_13, FID_c8_2_1, Id,...",True,True,False,True
9,Leif,14,"CCOS, COS_Reserv, COS_Seal, COS_Trap, Name, OB...",True,True,True,True


In [9]:
# ---------------------------------------------------------------------------
# Check combined COS arithmetic by dataset
# ---------------------------------------------------------------------------

candidate_fields = {
    "reservoir": [
        "COS_RES",
        "COS_Reserv",
        "Reservoir_",
        "Res_COS",
        "Horton_R_8",
    ],
    "seal": [
        "COS_SEAL",
        "COS_Seal",
        "Seal_COS",
        "Sea_COS",
        "Horton_Sea",
    ],
    "trap": [
        "COS_Trap",
        "Trap_COS",
        "Tr_COS",
        "Horton_Tra",
    ],
    "combined": [
        "Total_COS",
        "TotalCOS",
        "TCOS_CCUS",
        "CCOS",
        "Combined_C",
        "CCOS_CCUS",
        "Horton_TCO",
        "COS_CCUS",
    ],
}

def find_first_matching_field(columns, candidates):
    return next((field for field in candidates if field in columns), None)

cos_checks = []

for geological_group, directory in source_groups.items():

    for shp_path in sorted(directory.glob("*.shp")):

        gdf = gpd.read_file(shp_path)

        reservoir_field = find_first_matching_field(
            gdf.columns,
            candidate_fields["reservoir"],
        )
        seal_field = find_first_matching_field(
            gdf.columns,
            candidate_fields["seal"],
        )
        trap_field = find_first_matching_field(
            gdf.columns,
            candidate_fields["trap"],
        )
        combined_field = find_first_matching_field(
            gdf.columns,
            candidate_fields["combined"],
        )

        result = {
            "dataset": shp_path.stem,
            "reservoir_field": reservoir_field,
            "seal_field": seal_field,
            "trap_field": trap_field,
            "combined_field": combined_field,
            "formula_tested": None,
            "max_abs_error": None,
            "all_match": None,
        }

        if (
            reservoir_field
            and seal_field
            and combined_field
        ):

            if trap_field:

                calculated = (
                    gdf[reservoir_field]
                    * gdf[seal_field]
                    * gdf[trap_field]
                )

                result["formula_tested"] = (
                    "reservoir × seal × trap"
                )

            else:

                calculated = (
                    gdf[reservoir_field]
                    * gdf[seal_field]
                )

                result["formula_tested"] = (
                    "reservoir × seal"
                )

            error = (
                calculated - gdf[combined_field]
            ).abs()

            result["max_abs_error"] = error.max()
            result["all_match"] = bool(
                error.fillna(0).lt(1e-10).all()
            )

        cos_checks.append(result)

cos_check_df = pd.DataFrame(cos_checks)

display(
    cos_check_df
    .sort_values("dataset")
    .reset_index(drop=True)
)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,dataset,reservoir_field,seal_field,trap_field,combined_field,formula_tested,max_abs_error,all_match
0,ALBIAN_LOGAN_CANYON_TOTAL,COS_RES,COS_SEAL,NaN,Total_COS,reservoir × seal,5.551115e-17,True
1,BARREMIAN_UPPER_MISSISAUGA_TOTAL,COS_RES,COS_SEAL,NaN,Total_COS,reservoir × seal,1.110223e-16,True
2,BERRIASIAN_LOWER_MISSISAUGA_TOTAL,COS_RES,COS_SEAL,NaN,TotalCOS,reservoir × seal,5.551115e-17,True
3,Bjarni,COS_Reserv,COS_Seal,COS_Trap,TCOS_CCUS,reservoir × seal × trap,1.110223e-16,True
4,Cumberland_non_Magdalen,Res_COS,Sea_COS,Tr_COS,CCOS_CCUS,reservoir × seal × trap,5.551115e-17,True
5,Fundy,Reservoir_,Seal_COS,Trap_COS,Combined_C,reservoir × seal × trap,1.110223e-16,True
6,Gudrid,COS_Reserv,COS_Seal,COS_Trap,CCOS,reservoir × seal × trap,1.110223e-16,True
7,Horton_non_Magdalen,Horton_R_8,Horton_Sea,Horton_Tra,Horton_TCO,reservoir × seal × trap,5.551115e-17,True
8,LATE_ALBIAN_CREE_TOTAL,COS_RES,COS_SEAL,NaN,TotalCOS,reservoir × seal,5.551115e-17,True
9,Leif,COS_Reserv,COS_Seal,COS_Trap,CCOS,reservoir × seal × trap,5.000000e-07,False


In [10]:
# ---------------------------------------------------------------------------
# Investigate Pictou combined COS discrepancies
# ---------------------------------------------------------------------------

pictou = gpd.read_file(PALEO_DIR / "Pictou.shp")

pictou_check = pictou[
    [
        "COS_Reserv",
        "COS_Seal",
        "COS_Trap",
        "CCOS_CCUS",
        "Play_Des_2",
        "Play_Descr",
        "Play_des_1",
        "Descriptio",
    ]
].copy()

pictou_check["calculated_cos"] = (
    pictou_check["COS_Reserv"]
    * pictou_check["COS_Seal"]
    * pictou_check["COS_Trap"]
)

pictou_check["abs_error"] = (
    pictou_check["calculated_cos"]
    - pictou_check["CCOS_CCUS"]
).abs()

pictou_mismatch = (
    pictou_check[
        pictou_check["abs_error"] > 1e-10
    ]
    .sort_values("abs_error", ascending=False)
    .reset_index()
)

print(f"Rows with mismatch: {len(pictou_mismatch):,}")
print(
    "Maximum absolute error:",
    f"{pictou_mismatch['abs_error'].max():.6f}"
)

display(pictou_mismatch.head(25))

Rows with mismatch: 17
Maximum absolute error: 0.021000


,index,COS_Reserv,COS_Seal,COS_Trap,CCOS_CCUS,Play_Des_2,Play_Descr,Play_des_1,Descriptio,calculated_cos,abs_error
0,1165,0.1,0.30,0.1,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",High risk of migration to subcrop due to lack ...,Play Pictou structural/salt flank,0.0030,0.0210
1,1161,0.1,0.30,0.1,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",High risk of migration to subcrop due to lack ...,Play Pictou structural/salt flank,0.0030,0.0210
2,599,0.1,0.30,0.1,0.0150,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",High risk of migration to subcrop due to lack ...,Play Pictou structural/salt flank,0.0030,0.0120
3,585,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
4,586,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
5,1163,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
6,1162,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
7,589,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
8,605,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090
9,1164,0.1,0.30,0.5,0.0240,Cable Head Fm too shallow,"100 to 300 m Naufrage, low COS - could still h...",Risk of migration to subcrop through sand-domi...,Play Pictou structural/salt flank,0.0150,0.0090


In [11]:
# ---------------------------------------------------------------------------
# Confirm Leif discrepancy is only rounding precision
# ---------------------------------------------------------------------------

leif = gpd.read_file(MESO_DIR / "Leif.shp")

leif_calculated = (
    leif["COS_Reserv"]
    * leif["COS_Seal"]
    * leif["COS_Trap"]
)

leif_error = (
    leif_calculated - leif["CCOS"]
).abs()

print(f"Maximum absolute error: {leif_error.max():.10f}")
print(f"All rows within 1e-6:   {(leif_error <= 1e-6).all()}")

Maximum absolute error: 0.0000005000
All rows within 1e-6:   True


In [12]:
# ---------------------------------------------------------------------------
# Canonical source-field mapping
# ---------------------------------------------------------------------------

FIELD_MAPPINGS = {

    "ALBIAN_LOGAN_CANYON_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "Total_COS": "combined_cos",
    },

    "BARREMIAN_UPPER_MISSISAUGA_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "Total_COS": "combined_cos",
    },

    "BERRIASIAN_LOWER_MISSISAUGA_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "TotalCOS": "combined_cos",
    },

    "LATE_ALBIAN_CREE_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "TotalCOS": "combined_cos",
    },

    "UPPER_JURASSIC_MOHAWK_MIC_MAC_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "Total_COS": "combined_cos",
    },

    "VALANGINIAN_HAUTERIVIAN_MID_MISSISAUGA_TOTAL": {
        "COS_RES": "reservoir_cos",
        "COS_SEAL": "seal_cos",
        "TotalCOS": "combined_cos",
    },

    "Bjarni": {
        "COS_Reserv": "reservoir_cos",
        "Play100_Re": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play062_Se": "seal_description",
        "COS_Trap": "trap_cos",
        "Play100_Tr": "trap_description",
        "Type": "trap_type",
        "TCOS_CCUS": "combined_cos",
    },

    "Gudrid": {
        "COS_Reserv": "reservoir_cos",
        "Play053_Re": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play038_Se": "seal_description",
        "COS_Trap": "trap_cos",
        "Play053_Tr": "trap_description",
        "Type": "trap_type",
        "CCOS": "combined_cos",
        "Name": "assessment_area",
    },

    "Leif": {
        "COS_Reserv": "reservoir_cos",
        "Play038_Re": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play038_Se": "seal_description",
        "COS_Trap": "trap_cos",
        "Play038_Tr": "trap_description",
        "Type": "trap_type",
        "CCOS": "combined_cos",
        "Name": "assessment_area",
    },

    "Fundy": {
        "Reservoir_": "reservoir_cos",
        "Reservoir1": "reservoir_description",
        "Seal_COS": "seal_cos",
        "Seal_COS_D": "seal_description",
        "Trap_COS": "trap_cos",
        "Trap_COS_D": "trap_description",
        "Combined_C": "combined_cos",
    },

    "Cumberland_non_Magdalen": {
        "Res_COS": "reservoir_cos",
        "Res_COS_De": "reservoir_description",
        "Sea_COS": "seal_cos",
        "Sea_COS_De": "seal_description",
        "Tr_COS": "trap_cos",
        "Tr_COS_Des": "trap_description",
        "CCOS_CCUS": "combined_cos",
    },

    "Horton_non_Magdalen": {
        "Horton_R_8": "reservoir_cos",
        "Horton_R_9": "reservoir_description",
        "Horton_Sea": "seal_cos",
        "Horton_S_1": "seal_description",
        "Horton_Tra": "trap_cos",
        "Horton_T_1": "trap_description",
        "Horton_TCO": "combined_cos",
    },

    "Magdalen_Cumberland": {
        "COS_Reserv": "reservoir_cos",
        "Play312_De": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play_Descr": "seal_description",
        "COS_Trap": "trap_cos",
        "Play_des_1": "trap_description",
        "CCOS_CCUS": "combined_cos",
        "Descriptio": "play_description",
    },

    "Magdalen_Horton": {
        "COS_Reserv": "reservoir_cos",
        "Play352__2": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play352_De": "seal_description",
        "COS_Trap": "trap_cos",
        "Play352_Tr": "trap_description",
        "COS_CCUS": "combined_cos",
    },

    "Pictou": {
        "COS_Reserv": "reservoir_cos",
        "Play_Des_2": "reservoir_description",
        "COS_Seal": "seal_cos",
        "Play_Descr": "seal_description",
        "COS_Trap": "trap_cos",
        "Play_des_1": "trap_description",
        "CCOS_CCUS": "combined_cos",
        "Descriptio": "play_description",
    },
}

In [13]:
# ---------------------------------------------------------------------------
# Validate field mappings against source schemas
# ---------------------------------------------------------------------------

mapping_validation = []

for geological_group, directory in source_groups.items():

    for shp_path in sorted(directory.glob("*.shp")):

        dataset = shp_path.stem
        gdf = gpd.read_file(shp_path)

        mapping = FIELD_MAPPINGS.get(dataset, {})

        for source_field, canonical_field in mapping.items():

            mapping_validation.append(
                {
                    "dataset": dataset,
                    "source_field": source_field,
                    "canonical_field": canonical_field,
                    "source_field_exists": source_field in gdf.columns,
                }
            )

mapping_validation_df = pd.DataFrame(mapping_validation)

display(mapping_validation_df)

print(
    "\nMissing mapped fields:",
    (~mapping_validation_df["source_field_exists"]).sum(),
)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,dataset,source_field,canonical_field,source_field_exists
0,ALBIAN_LOGAN_CANYON_TOTAL,COS_RES,reservoir_cos,True
1,ALBIAN_LOGAN_CANYON_TOTAL,COS_SEAL,seal_cos,True
2,ALBIAN_LOGAN_CANYON_TOTAL,Total_COS,combined_cos,True
3,BARREMIAN_UPPER_MISSISAUGA_TOTAL,COS_RES,reservoir_cos,True
4,BARREMIAN_UPPER_MISSISAUGA_TOTAL,COS_SEAL,seal_cos,True
...,...,...,...,...
83,Pictou,Play_Descr,seal_description,True
84,Pictou,COS_Trap,trap_cos,True
85,Pictou,Play_des_1,trap_description,True
86,Pictou,CCOS_CCUS,combined_cos,True



Missing mapped fields: 0


In [14]:
# ---------------------------------------------------------------------------
# Define Silver dataset metadata
# ---------------------------------------------------------------------------

SUBMISSION_DATE = "20260911"
ACTIVITY_CODE = "13"
CREATOR_INITIALS = "AV"

SILVER_METADATA = {
    # -----------------------------------------------------------------------
    # Dataset identity and classification
    # -----------------------------------------------------------------------

    "title": "Atlantic Canada CO2 Storage Chance of Success",
    "dataset_role": "geological_prospectivity",
    "assessment_type": "chance_of_success",
    "capacity_data": False,

    # -----------------------------------------------------------------------
    # Source provenance
    # -----------------------------------------------------------------------

    "source_title": (
        "Preliminary assessment of geological carbon-storage potential "
        "of Atlantic Canada"
    ),
    "source_publication": (
        "Geological Survey of Canada Open File 8996"
    ),
    "source_year": 2023,
    "source_doi": "10.4095/332145",
    "source_url": PUBLICATION_URL,

    # -----------------------------------------------------------------------
    # CanCO2Re minimum metadata: Who / What / When / Where / How
    # -----------------------------------------------------------------------

    "who": "Andrew Vigars / CanCO2Re Activity 13",

    "what": (
        "Harmonized regional geological CO2 storage Chance of Success "
        "(COS) mapping for Atlantic Canada. The dataset contains reservoir, "
        "seal, trap where available, and combined COS attributes. It "
        "represents geological prospectivity and does not represent "
        "quantified storage capacity, injectivity, permitted storage "
        "resource, or project-ready storage capacity."
    ),

    "when": "2026-09-11",

    "where": (
        "Atlantic Canada; NAD83 / Canada Atlas Lambert, EPSG:3978"
    ),

    "how": (
        "Derived from 15 Geological Survey of Canada Open File 8996 "
        "shapefiles. Source schemas were harmonized into a common COS "
        "schema, source features were appended without dissolving geometry, "
        "source provenance was retained, geometries were reprojected to "
        "EPSG:3978, feature areas were recalculated, and published combined "
        "COS values were independently reconstructed for QA. Source "
        "discrepancies were flagged but not corrected."
    ),

    # -----------------------------------------------------------------------
    # Additional descriptive metadata
    # -----------------------------------------------------------------------

    "credits": (
        "Source geological assessment: Geological Survey of Canada. "
        "Silver-layer harmonization: Andrew Vigars, CanCO2Re Activity 13."
    ),

    "use_limitations": (
        "Chance of Success values represent regional geological "
        "prospectivity. They must not be interpreted as quantified CO2 "
        "storage capacity, injectivity, permitted storage resource, or "
        "project-ready storage capacity."
    ),

    "keywords": (
        "carbon storage; CO2 storage; CCUS; geological storage; "
        "chance of success; prospectivity; Atlantic Canada; CanCO2Re"
    ),

    # -----------------------------------------------------------------------
    # Processing / submission metadata
    # -----------------------------------------------------------------------

    "silver_crs": "EPSG:3978",
    "bronze_layer_count": 15,
    "canco2re_activity": ACTIVITY_CODE,
    "creator_initials": CREATOR_INITIALS,
}

In [15]:
# ---------------------------------------------------------------------------
# Prepare formal GeoPackage metadata
# ---------------------------------------------------------------------------

GPKG_DATASET_METADATA = {
    "TITLE": SILVER_METADATA["title"],
    "CREATOR": SILVER_METADATA["who"],
    "DATE": SILVER_METADATA["when"],
    "SOURCE": SILVER_METADATA["source_publication"],
    "DOI": SILVER_METADATA["source_doi"],
    "CRS": SILVER_METADATA["silver_crs"],
}

GPKG_LAYER_METADATA = {
    "TITLE": SILVER_METADATA["title"],
    "ABSTRACT": SILVER_METADATA["what"],
    "DESCRIPTION": SILVER_METADATA["what"],
    "CONTACT": SILVER_METADATA["who"],
    "CREDITS": SILVER_METADATA["credits"],
    "USE_LIMITATIONS": SILVER_METADATA["use_limitations"],
    "KEYWORDS": SILVER_METADATA["keywords"],
    "SOURCE": SILVER_METADATA["source_publication"],
    "SOURCE_DOI": SILVER_METADATA["source_doi"],
    "PROCESSING": SILVER_METADATA["how"],
    "GEOGRAPHIC_EXTENT": SILVER_METADATA["where"],
}

In [16]:
# ---------------------------------------------------------------------------
# Build harmonized Silver GeoDataFrame
# ---------------------------------------------------------------------------

harmonized_layers = []

for geological_group, directory in source_groups.items():

    for shp_path in sorted(directory.glob("*.shp")):

        dataset = shp_path.stem
        gdf = gpd.read_file(shp_path)

        mapping = FIELD_MAPPINGS[dataset]

        keep_fields = list(mapping.keys()) + ["geometry"]

        harmonized = (
            gdf[keep_fields]
            .rename(columns=mapping)
            .copy()
        )

        # -------------------------------------------------------------------
        # Feature provenance
        # -------------------------------------------------------------------

        harmonized["source_dataset"] = dataset
        harmonized["source_feature_id"] = range(len(harmonized))
        harmonized["source_feature_uid"] = (
            harmonized["source_dataset"]
            + ":"
            + harmonized["source_feature_id"].astype(str)
        )
        harmonized["source_crs"] = gdf.crs.to_string()

        # -------------------------------------------------------------------
        # Dataset classification
        # -------------------------------------------------------------------

        harmonized["geological_group"] = geological_group
        harmonized["assessment_type"] = (
            SILVER_METADATA["assessment_type"]
        )
        harmonized["data_class"] = (
            SILVER_METADATA["dataset_role"]
        )
        harmonized["capacity_data"] = (
            SILVER_METADATA["capacity_data"]
        )

        # -------------------------------------------------------------------
        # Standardize spatial reference
        # -------------------------------------------------------------------

        harmonized = harmonized.to_crs(CANCO2RE_CRS)

        harmonized["area_km2"] = (
            harmonized.geometry.area / 1_000_000
        )

        harmonized_layers.append(harmonized)


# ---------------------------------------------------------------------------
# Combine harmonized source layers
# ---------------------------------------------------------------------------

atlantic_cos = pd.concat(
    harmonized_layers,
    ignore_index=True,
)

atlantic_cos = gpd.GeoDataFrame(
    atlantic_cos,
    geometry="geometry",
    crs=CANCO2RE_CRS,
)


# ---------------------------------------------------------------------------
# Preview
# ---------------------------------------------------------------------------

display(atlantic_cos.head())

print(f"Features: {len(atlantic_cos):,}")
print(f"CRS:      {atlantic_cos.crs}")

print("\nColumns:")
print(atlantic_cos.columns.tolist())

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\Mesozoic-Cenozoic COS Mapping\Bjarni.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


,reservoir_cos,seal_cos,combined_cos,geometry,source_dataset,source_feature_id,source_feature_uid,source_crs,geological_group,assessment_type,data_class,capacity_data,area_km2,reservoir_description,seal_description,trap_cos,trap_description,trap_type,assessment_area,play_description
0,0.75,0.25,0.1875,"POLYGON ((2669686.105 147188.654, 2683111.377 ...",ALBIAN_LOGAN_CANYON_TOTAL,0,ALBIAN_LOGAN_CANYON_TOTAL:0,EPSG:26720,Mesozoic-Cenozoic,chance_of_success,geological_prospectivity,False,10487.852540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.75,0.75,0.5625,"POLYGON ((2905684.474 379622.485, 2900221.161 ...",ALBIAN_LOGAN_CANYON_TOTAL,1,ALBIAN_LOGAN_CANYON_TOTAL:1,EPSG:26720,Mesozoic-Cenozoic,chance_of_success,geological_prospectivity,False,27473.055505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.75,0.50,0.3750,"POLYGON ((2861701.35 401494.282, 2829381.778 3...",ALBIAN_LOGAN_CANYON_TOTAL,2,ALBIAN_LOGAN_CANYON_TOTAL:2,EPSG:26720,Mesozoic-Cenozoic,chance_of_success,geological_prospectivity,False,9390.660676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.45,0.15,0.0675,"MULTIPOLYGON (((2654101.719 38230.803, 2662998...",ALBIAN_LOGAN_CANYON_TOTAL,3,ALBIAN_LOGAN_CANYON_TOTAL:3,EPSG:26720,Mesozoic-Cenozoic,chance_of_success,geological_prospectivity,False,126.635065,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.45,0.75,0.3375,"POLYGON ((2917046.365 373985.856, 2923058.282 ...",ALBIAN_LOGAN_CANYON_TOTAL,4,ALBIAN_LOGAN_CANYON_TOTAL:4,EPSG:26720,Mesozoic-Cenozoic,chance_of_success,geological_prospectivity,False,67383.863222,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Features: 4,711
CRS:      EPSG:3978

Columns:
['reservoir_cos', 'seal_cos', 'combined_cos', 'geometry', 'source_dataset', 'source_feature_id', 'source_feature_uid', 'source_crs', 'geological_group', 'assessment_type', 'data_class', 'capacity_data', 'area_km2', 'reservoir_description', 'seal_description', 'trap_cos', 'trap_description', 'trap_type', 'assessment_area', 'play_description']


In [17]:
# ---------------------------------------------------------------------------
# Classify dataset-level COS arithmetic validation
# ---------------------------------------------------------------------------

cos_validation_summary = cos_check_df.copy()

cos_validation_summary["validation_status"] = "exact_match"

cos_validation_summary.loc[
    (cos_validation_summary["max_abs_error"] > 0)
    & (cos_validation_summary["max_abs_error"] <= 1e-6),
    "validation_status",
] = "rounding_tolerance"

cos_validation_summary.loc[
    cos_validation_summary["max_abs_error"] > 1e-6,
    "validation_status",
] = "source_discrepancy"

display(
    cos_validation_summary[
        [
            "dataset",
            "formula_tested",
            "max_abs_error",
            "validation_status",
        ]
    ]
    .sort_values("dataset")
    .reset_index(drop=True)
)

,dataset,formula_tested,max_abs_error,validation_status
0,ALBIAN_LOGAN_CANYON_TOTAL,reservoir × seal,5.551115e-17,rounding_tolerance
1,BARREMIAN_UPPER_MISSISAUGA_TOTAL,reservoir × seal,1.110223e-16,rounding_tolerance
2,BERRIASIAN_LOWER_MISSISAUGA_TOTAL,reservoir × seal,5.551115e-17,rounding_tolerance
3,Bjarni,reservoir × seal × trap,1.110223e-16,rounding_tolerance
4,Cumberland_non_Magdalen,reservoir × seal × trap,5.551115e-17,rounding_tolerance
5,Fundy,reservoir × seal × trap,1.110223e-16,rounding_tolerance
6,Gudrid,reservoir × seal × trap,1.110223e-16,rounding_tolerance
7,Horton_non_Magdalen,reservoir × seal × trap,5.551115e-17,rounding_tolerance
8,LATE_ALBIAN_CREE_TOTAL,reservoir × seal,5.551115e-17,rounding_tolerance
9,Leif,reservoir × seal × trap,5.000000e-07,rounding_tolerance


In [18]:
# ---------------------------------------------------------------------------
# Final harmonized-layer QA summary by source dataset
# ---------------------------------------------------------------------------

qa_summary = (
    atlantic_cos
    .groupby(
        [
            "geological_group",
            "source_dataset",
        ],
        dropna=False,
    )
    .agg(
        features=("source_feature_id", "count"),
        reservoir_cos_min=("reservoir_cos", "min"),
        reservoir_cos_max=("reservoir_cos", "max"),
        seal_cos_min=("seal_cos", "min"),
        seal_cos_max=("seal_cos", "max"),
        trap_cos_min=("trap_cos", "min"),
        trap_cos_max=("trap_cos", "max"),
        combined_cos_min=("combined_cos", "min"),
        combined_cos_max=("combined_cos", "max"),
        total_area_km2=("area_km2", "sum"),
    )
    .reset_index()
)

qa_summary = qa_summary.merge(
    cos_validation_summary[
        [
            "dataset",
            "formula_tested",
            "max_abs_error",
            "validation_status",
        ]
    ],
    left_on="source_dataset",
    right_on="dataset",
    how="left",
).drop(columns="dataset")

display(qa_summary)

,geological_group,source_dataset,features,reservoir_cos_min,reservoir_cos_max,seal_cos_min,seal_cos_max,trap_cos_min,trap_cos_max,combined_cos_min,combined_cos_max,total_area_km2,formula_tested,max_abs_error,validation_status
0,Mesozoic-Cenozoic,ALBIAN_LOGAN_CANYON_TOTAL,17,0.15,0.75,0.1500,0.7500,NaN,NaN,0.030000,0.562500,162761.597961,reservoir × seal,5.551115e-17,rounding_tolerance
1,Mesozoic-Cenozoic,BARREMIAN_UPPER_MISSISAUGA_TOTAL,10,0.15,0.70,0.2500,0.7500,NaN,NaN,0.100000,0.525000,163244.496792,reservoir × seal,1.110223e-16,rounding_tolerance
2,Mesozoic-Cenozoic,BERRIASIAN_LOWER_MISSISAUGA_TOTAL,12,0.25,0.75,0.2500,0.7500,NaN,NaN,0.062500,0.562500,162762.013129,reservoir × seal,5.551115e-17,rounding_tolerance
3,Mesozoic-Cenozoic,Bjarni,407,0.01,1.00,0.0100,0.9000,0.01,0.90,0.000001,0.810000,286733.809652,reservoir × seal × trap,1.110223e-16,rounding_tolerance
4,Mesozoic-Cenozoic,Fundy,43,0.05,0.80,0.1000,0.8000,0.10,0.80,0.000500,0.512000,22496.554829,reservoir × seal × trap,1.110223e-16,rounding_tolerance
5,Mesozoic-Cenozoic,Gudrid,812,0.01,0.95,0.0100,0.9500,0.01,0.90,0.000001,0.767125,286733.754550,reservoir × seal × trap,1.110223e-16,rounding_tolerance
6,Mesozoic-Cenozoic,LATE_ALBIAN_CREE_TOTAL,8,0.15,0.65,0.1500,0.6500,NaN,NaN,0.022500,0.422500,162767.130635,reservoir × seal,5.551115e-17,rounding_tolerance
7,Mesozoic-Cenozoic,Leif,472,0.01,0.95,0.0199,0.9975,0.01,0.95,0.000002,0.757350,286733.562900,reservoir × seal × trap,5.000000e-07,rounding_tolerance
8,Mesozoic-Cenozoic,UPPER_JURASSIC_MOHAWK_MIC_MAC_TOTAL,17,0.35,0.75,0.3500,0.7500,NaN,NaN,0.122500,0.562500,162766.678988,reservoir × seal,5.551115e-17,rounding_tolerance
9,Mesozoic-Cenozoic,VALANGINIAN_HAUTERIVIAN_MID_MISSISAUGA_TOTAL,13,0.30,0.75,0.5500,0.7000,NaN,NaN,0.165000,0.525000,164456.101269,reservoir × seal,1.110223e-16,rounding_tolerance


## Silver export

The harmonized Atlantic Canada COS dataset is exported as a GeoPackage
for use within the CanCO₂Re GIS workflow.

The exported Silver layer:

- preserves all 4,711 source-derived assessment features;
- uses the harmonized COS schema developed in this notebook;
- retains source dataset and source CRS provenance;
- stores geometry in the CanCO₂Re project CRS, EPSG:3978;
- includes dataset classification fields identifying the layer as geological prospectivity;
- preserves published COS values while storing QA-derived calculations separately;
- represents geological prospectivity rather than quantified storage capacity.

No source polygons are dissolved or spatially unioned during export.

In [19]:
# ---------------------------------------------------------------------------
# Export harmonized Atlantic Canada COS Silver GeoPackage
# ---------------------------------------------------------------------------

EXPORT_DIR = ROOT / "derived"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

GPKG_FILENAME = (
    f"{SUBMISSION_DATE}_"
    f"{ACTIVITY_CODE}_"
    f"AtlanticStorageCOS_"
    f"{CREATOR_INITIALS}.gpkg"
)

GPKG_PATH = EXPORT_DIR / GPKG_FILENAME

# Rebuild cleanly
if GPKG_PATH.exists():
    GPKG_PATH.unlink()

# ---------------------------------------------------------------------------
# Export spatial Silver layer with formal metadata
# ---------------------------------------------------------------------------

atlantic_cos.to_file(
    GPKG_PATH,
    layer="atlantic_cos",
    driver="GPKG",
    engine="pyogrio",
    dataset_metadata=GPKG_DATASET_METADATA,
    layer_metadata=GPKG_LAYER_METADATA,
)

# ---------------------------------------------------------------------------
# Prepare metadata table
# ---------------------------------------------------------------------------

metadata_df = pd.DataFrame(
    [
        {"key": key, "value": str(value)}
        for key, value in SILVER_METADATA.items()
    ]
)

# ---------------------------------------------------------------------------
# Export metadata and QA tables
# ---------------------------------------------------------------------------

with sqlite3.connect(GPKG_PATH) as conn:

    metadata_df.to_sql(
        "metadata_atlantic_cos",
        conn,
        if_exists="replace",
        index=False,
    )

    qa_summary.to_sql(
        "qa_atlantic_cos",
        conn,
        if_exists="replace",
        index=False,
    )

    # -----------------------------------------------------------------------
    # Register non-spatial tables as GeoPackage attribute tables
    # -----------------------------------------------------------------------

    attribute_tables = {
        "metadata_atlantic_cos": (
            "Atlantic COS dataset metadata"
        ),
        "qa_atlantic_cos": (
            "Atlantic COS source-dataset QA summary"
        ),
    }

    for table_name, description in attribute_tables.items():

        # Remove stale registration if this notebook is rerun
        conn.execute(
            """
            DELETE FROM gpkg_contents
            WHERE table_name = ?
            """,
            (table_name,),
        )

        conn.execute(
            """
            INSERT INTO gpkg_contents (
                table_name,
                data_type,
                identifier,
                description,
                last_change,
                min_x,
                min_y,
                max_x,
                max_y,
                srs_id
            )
            VALUES (
                ?,
                'attributes',
                ?,
                ?,
                strftime(
                    '%Y-%m-%dT%H:%M:%fZ',
                    'now'
                ),
                NULL,
                NULL,
                NULL,
                NULL,
                NULL
            )
            """,
            (
                table_name,
                table_name,
                description,
            ),
        )

    conn.commit()

# ---------------------------------------------------------------------------
# Export summary
# ---------------------------------------------------------------------------

print(f"Exported: {GPKG_PATH}")
print(f"Features: {len(atlantic_cos):,}")
print(f"CRS:      {atlantic_cos.crs}")
print(f"Columns:  {len(atlantic_cos.columns)}")

print("\nGeoPackage contents:")
print("- atlantic_cos")
print("- metadata_atlantic_cos")
print("- qa_atlantic_cos")

Exported: C:\Users\aviga\Research\potential data\Storage\Atlantic Canada CO2 Storage\derived\20260911_13_AtlanticStorageCOS_AV.gpkg
Features: 4,711
CRS:      EPSG:3978
Columns:  20

GeoPackage contents:
- atlantic_cos
- metadata_atlantic_cos
- qa_atlantic_cos


In [20]:
# ---------------------------------------------------------------------------
# Validate GeoPackage export
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Validate spatial layer
# ---------------------------------------------------------------------------

atlantic_cos_check = gpd.read_file(
    GPKG_PATH,
    layer="atlantic_cos",
)

assert len(atlantic_cos_check) == len(atlantic_cos)
assert atlantic_cos_check.crs == atlantic_cos.crs

# Compare non-geometry attribute columns
source_attribute_columns = {
    col
    for col in atlantic_cos.columns
    if col != atlantic_cos.geometry.name
}

export_attribute_columns = {
    col
    for col in atlantic_cos_check.columns
    if col != atlantic_cos_check.geometry.name
}

missing_columns = (
    source_attribute_columns
    - export_attribute_columns
)

extra_columns = (
    export_attribute_columns
    - source_attribute_columns
)

print(
    "Geometry column before export:",
    atlantic_cos.geometry.name,
)

print(
    "Geometry column after export: ",
    atlantic_cos_check.geometry.name,
)

print("\nMissing attribute columns:")
print(sorted(missing_columns))

print("\nExtra attribute columns:")
print(sorted(extra_columns))

assert not missing_columns
assert not extra_columns


# ---------------------------------------------------------------------------
# Validate non-spatial metadata and QA tables
# ---------------------------------------------------------------------------

with sqlite3.connect(GPKG_PATH) as conn:

    metadata_check = pd.read_sql(
        "SELECT * FROM metadata_atlantic_cos",
        conn,
    )

    qa_check = pd.read_sql(
        "SELECT * FROM qa_atlantic_cos",
        conn,
    )

    table_names = pd.read_sql(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn,
    )["name"].tolist()

assert len(metadata_check) == len(SILVER_METADATA)
assert len(qa_check) == len(qa_summary)


# ---------------------------------------------------------------------------
# Check formal GeoPackage metadata extension
# ---------------------------------------------------------------------------

has_gpkg_metadata = (
    "gpkg_metadata" in table_names
)

has_gpkg_metadata_reference = (
    "gpkg_metadata_reference" in table_names
)

print("\nFormal GeoPackage metadata:")
print(f"gpkg_metadata present:           {has_gpkg_metadata}")
print(
    "gpkg_metadata_reference present:",
    has_gpkg_metadata_reference,
)

if (
    has_gpkg_metadata
    and has_gpkg_metadata_reference
):

    with sqlite3.connect(GPKG_PATH) as conn:

        formal_metadata_check = pd.read_sql(
            "SELECT * FROM gpkg_metadata",
            conn,
        )

        metadata_reference_check = pd.read_sql(
            "SELECT * FROM gpkg_metadata_reference",
            conn,
        )

    print(
        "Formal metadata records:       ",
        len(formal_metadata_check),
    )

    print(
        "Metadata reference records:    ",
        len(metadata_reference_check),
    )


# ---------------------------------------------------------------------------
# Validation summary
# ---------------------------------------------------------------------------

print("\nGeoPackage export validated successfully.")

print("\nSpatial layer:")
print(
    f"Features written: {len(atlantic_cos_check):,}"
)
print(
    f"CRS:              {atlantic_cos_check.crs}"
)
print(
    f"Attributes:       {len(export_attribute_columns)}"
)

print("\nMetadata table:")
print(
    f"Rows written:     {len(metadata_check):,}"
)

print("\nQA table:")
print(
    f"Rows written:     {len(qa_check):,}"
)

Geometry column before export: geometry
Geometry column after export:  geometry

Missing attribute columns:
[]

Extra attribute columns:
[]

Formal GeoPackage metadata:
gpkg_metadata present:           True
gpkg_metadata_reference present: True
Formal metadata records:        2
Metadata reference records:     2

GeoPackage export validated successfully.

Spatial layer:
Features written: 4,711
CRS:              EPSG:3978
Attributes:       19

Metadata table:
Rows written:     21

QA table:
Rows written:     15


### Spatial interpretation

Many COS datasets occupy substantially overlapping assessment footprints.

Similar total mapped areas across formations indicate that individual
layers represent alternative or vertically stacked geological plays
evaluated over common regional extents, rather than mutually exclusive
storage areas.

Accordingly, mapped areas should not be summed across datasets to infer
total geological storage area or storage capacity.

## Exploration conclusions

The Geological Survey of Canada Open File 8996 GIS package contains
15 regional geological CO₂ storage Chance of Success (COS) layers.

The source data represent geological prospectivity rather than quantified
CO₂ storage capacity.

COS values assess reservoir, seal, and where applicable trap conditions.
Most layers calculate combined COS as:

`reservoir COS × seal COS × trap COS`

while five Scotian Margin datasets use:

`reservoir COS × seal COS`

because no explicit trap attribute is included in those source layers.

The source schemas vary substantially but can be harmonized into a common
prospectivity schema containing reservoir, seal, trap, combined COS,
descriptive geological rationale, source provenance, and geometry.

Arithmetic validation found:
- exact agreement for most source datasets;
- only rounding-level differences for Leif;
- 17 Pictou records with larger source discrepancies, which should be
  preserved and flagged rather than automatically corrected.

All harmonized Silver geometries were standardized to EPSG:3978
(NAD83 / Canada Atlas Lambert), the CanCO₂Re project standard CRS, for
comparison and area calculation. Original Bronze CRS information is retained
in `source_crs` for provenance.

These datasets should be treated as geological prospectivity/screening
information. They do not provide quantified storage capacity, injectivity,
or project-ready storage resource estimates.

Quantified storage-capacity exploration will be handled separately,
beginning with NATCARB and other sources containing explicit volumetric
or mass-based storage-resource estimates.